In [1]:
import requests
import json
import time
import pandas as pd
from pathlib import Path

In [2]:
raw_dir = Path("data/raw")
raw_dir.mkdir(parents=True, exist_ok=True)

In [3]:
# Fetch data from API
def fetch_json(endpoint, retries=4, **params):
    url = f"https://api.openf1.org/v1/{endpoint}"
    for attempt in range(retries):
        try:
    
            response = requests.get(url, params=params)
            response.raise_for_status()
            time.sleep(0.25)
            return response.json()
        except requests.exceptions.HTTPError:
            if attempt == retries - 1:
                raise
            time.sleep(2 ** attempt)

In [4]:
# Check wether cached file already exists, if not then fetch from API
def cached_fetch(endpoint, session_key):
    path = raw_dir / f"{endpoint}_{session_key}.json"
    if path.exists():
        with open(path) as f:
            return json.load(f)
        
    data = fetch_json(endpoint, session_key=session_key)
    with open(path, "w") as f:
        json.dump(data, f)
    return data

In [5]:
years = [2023, 2024, 2025]
races = []

for year in years:
    races += fetch_json("sessions", year=year, session_name="Race") # Get all Races from the seasons 2023 - 2025
    
df = pd.DataFrame(races)

sessions = df[~df['is_cancelled']]['session_key'].to_list() # save all session_keys except the ones that got cancelled

In [6]:
# get the intervals opened with start message and closed with end message, if there is no end message, either red flag or the session end closes the intervall
def active_intervals(race_control, start_messages, end_messages, session_end):
    race_control_start = race_control[race_control["message"].isin(start_messages)]
    race_control_end = race_control[race_control["message"].isin(end_messages)]

    intervals = []
    starts = race_control_start.sort_values("date")["date"]
    ends = race_control_end.sort_values("date")["date"]

    for start in starts:
        later_ends = ends[ends > start]
        end = later_ends.iloc[0] if len(later_ends) else session_end
        intervals.append((start, end))
    return intervals

In [7]:
# check if the interval is inside of a lap, plauibillity check
def touches_any(lap_start, lap_end, intervals):
    for (s, e) in intervals:
        if lap_start <= e and lap_end >= s:
            return True
    return False

In [8]:
def build_session(session_key):
    
    # get data from all 
    laps = pd.DataFrame(cached_fetch("laps", session_key)) 
    stints = pd.DataFrame(cached_fetch("stints", session_key))
    weather = pd.DataFrame(cached_fetch("weather", session_key))
    drivers = pd.DataFrame(cached_fetch("drivers", session_key))
    race_control = pd.DataFrame(cached_fetch("race_control", session_key))

    laps = laps.dropna(subset=["date_start"])

    stints = stints.dropna(subset="lap_start")
    stints["lap_start"] = stints["lap_start"].astype(int)
    stints["lap_end"] = stints["lap_end"].astype(int)   
    
    laps   = laps.sort_values("lap_number")

    # Get rid of session_key and meeting_key
    stint_cols = ["driver_number", "stint_number", "lap_start", "lap_end", "compound", "tyre_age_at_start"]
    stints = stints[stint_cols]

    stints = stints.sort_values("lap_start")

    # Merge stints into laps over the sting range (stint_1: lap_start -> stint_2: lap_start)
    merged = pd.merge_asof(
        laps,
        stints,
        left_on="lap_number",
        right_on="lap_start",
        by="driver_number",
        direction="backward",
    )

    weather_cols = ["date", "air_temperature", "track_temperature", "humidity", "rainfall"]
    weather = weather[weather_cols].copy()
    
    merged["date_start"] = pd.to_datetime(merged["date_start"], format="ISO8601")
    weather["date"] = pd.to_datetime(weather["date"], format="ISO8601")
    race_control["date"] = pd.to_datetime(race_control["date"], format="ISO8601")
    
    merged = merged.sort_values("date_start")
    weather = weather.sort_values("date")
    
    merged = pd.merge_asof(
        merged, weather,
        left_on="date_start", right_on="date",
        direction="nearest",
    )

    drivers = drivers[["driver_number", "team_name"]]
    merged = merged.merge(drivers, on="driver_number", how="left")
    
    session_meta = df[["session_key", "circuit_key", "year"]]
    merged = merged.merge(session_meta, on="session_key", how="left")


    # Create new columns
    merged["stint_age"] = merged["lap_number"] - merged["lap_start"] + 1
    merged["tyre_age_total"] = merged["tyre_age_at_start"] + merged["stint_age"]
    merged["stint_id"] = merged["driver_number"].astype(str) + "_" + merged["session_key"].astype(str)  + "_" + merged["stint_number"].astype(str)

    # flag all laps with safety car
    session_end = merged["date_start"].max()
    sc_periods = active_intervals(race_control, ["SAFETY CAR DEPLOYED", "SAFETY CAR THROUGH THE PIT LANE"], ["SAFETY CAR IN THIS LAP", "RED FLAG"], session_end)
    merged["is_under_sc"] = merged.apply(
       lambda row: touches_any(row["date_start"], row["date_start"] + pd.to_timedelta(row["lap_duration"], unit="s"), sc_periods),
       axis=1
    )

    # Flag all laps with virtual safety car
    vsc_periods = active_intervals(race_control, ["VIRTUAL SAFETY CAR DEPLOYED"], ["VIRTUAL SAFETY CAR ENDING", "RED FLAG"], session_end)
    merged["is_under_vsc"] = merged.apply(
        lambda row: touches_any(row["date_start"], row["date_start"] + pd.to_timedelta(row["lap_duration"], unit="s"), vsc_periods),
        axis=1
    )

    # Flag all laps with yellow flag
    yellow_times = race_control[race_control["flag"].isin(["YELLOW", "DOUBLE YELLOW"])]["date"]
    yellow_periods = [(t, t) for t in yellow_times]
    merged["is_under_yellow"] = merged.apply(
        lambda row: touches_any(row["date_start"], row["date_start"] + pd.to_timedelta(row["lap_duration"], unit="s"), yellow_periods),
        axis=1
    )

    # Flag all laps with red flag
    red_times = race_control[race_control["flag"] == "RED"]["date"]
    red_periods = [(t, t) for t in red_times]
    merged["is_red_flag_lap"] = merged.apply(
        lambda row: touches_any(row["date_start"], row["date_start"] + pd.to_timedelta(row["lap_duration"], unit="s"), red_periods),
        axis=1
    )

    # get the In-Laps
    merged["driver_max_stint"] = merged.groupby("driver_number")["stint_number"].transform("max")
    merged["is_in_lap"] = (merged["lap_number"] == merged["lap_end"]) & (merged["stint_number"] < merged["driver_max_stint"])


    return merged


In [9]:
frames = [build_session(session) for session in sessions]

dataset = pd.concat(frames, ignore_index=True)

In [10]:
dataset = dataset.rename(columns={"circuit_key": "track_id", "team_name": "team_id"})

In [11]:
c_codes = pd.read_csv("data/tyre_c_codes.csv")
c_codes = c_codes[c_codes["session_type"] == "Race"][["session_key", "compound", "c_code"]]
c_codes = c_codes.rename(columns={"c_code": "c_compound"})
dataset = dataset.merge(c_codes, on=["session_key", "compound"], how="left")

In [12]:
keep = [
    "session_key", 
    "driver_number", 
    "lap_number", 
    "stint_number", 
    "stint_id",
    "lap_start", 
    "lap_end",
    "date_start", 
    "lap_duration",
    "compound",
    "c_compound",
    "tyre_age_at_start", 
    "stint_age", 
    "tyre_age_total",
    "track_id", 
    "team_id", 
    "year",
    "air_temperature", 
    "track_temperature", 
    "humidity", 
    "rainfall",
    "is_pit_out_lap", 
    "is_in_lap", 
    "is_under_sc", 
    "is_under_vsc", 
    "is_under_yellow", 
    "is_red_flag_lap",
]
dataset = dataset[keep].copy()


In [13]:
dataset["humidity"].value_counts(dropna=False)

humidity
60.0    2866
64.0    2358
21.0    2338
61.0    2172
62.0    2153
        ... 
17.0      46
86.0      30
88.0      29
87.0      19
92.0      14
Name: count, Length: 76, dtype: int64

In [14]:
dataset.to_csv("data/dataset.csv", index=False)